# Kod za minio S3 buckete
Prvo se definira .yml

In [4]:
%%writefile ../minio.docker-compose.yml

version: '3'

networks: #TODO: remove this after merging with main docker-compose
  cv-network:
    name: "cv-network"
    driver: bridge

volumes:
  minio_data:

services:
  minio:
      image: minio/minio:latest
      container_name: minio
      networks:
        - cv-network
      env_file:
          -  ./S3Storage/minio.env
          -  ./S3Storage/minio.api-voditelj.shared.env
      volumes:
        - minio_data:/data
      ports:
        - "${MINIO_API_PORT:-9901}:9901"
        - "${MINIO_CONSOLE_PORT:-9902}:9902"
      command: server /data --address ":9901" --console-address ":9902"

Overwriting ../minio.docker-compose.yml


# Definiranje .env file-a

In [ ]:
%%writefile minio.env
MINIO_ROOT_USER=minioadmin
MINIO_ROOT_PASSWORD=minioadmin



Writing minio.env


Definiranje Shared .env file-a

In [2]:
%%writefile minio.api-voditelj.shared.env
APP_MINIO_ACCESS_KEY=apivoditeljuser
APP_MINIO_SECRET_KEY=apivoditeljuserPass
MINIO_ENDPOINT=minio:9901
MINIO_SECURE=FALSE

Overwriting minio.api-voditelj.shared.env


# Dodavanje minia u .env da ih docker compose ucita

In [35]:
import os

def update_compose_file_env(env_file_path, compose_file_name):
    """
    Updates the .env file to include the specified compose file in the COMPOSE_FILE variable.

    Args:
        env_file_path (str): The path to the .env file.
        compose_file_name (str): The name of the compose file to add.
    """
    try:
        with open(env_file_path, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: .env file not found at {env_file_path}")
        return

    updated = False
    with open(env_file_path, 'w') as f:
        for line in lines:
            if line.startswith("COMPOSE_FILE="):
                if compose_file_name not in line:
                    line = line.strip()
                    if line.endswith("="):
                        line += compose_file_name + "\n"
                    else:
                        line += ":" + compose_file_name + "\n"
                    updated = True
            f.write(line)

    if updated:
        print(f"Updated COMPOSE_FILE in {env_file_path}")
    else:
        print(f"COMPOSE_FILE already contains {compose_file_name} in {env_file_path}")

if __name__ == "__main__":
    env_file = "../.env"
    compose_file = "minio.docker-compose.yml"
    update_compose_file_env(env_file, compose_file)

COMPOSE_FILE already contains minio.docker-compose.yml in ../.env


Azururaj docker compose za manager app

In [1]:

from ruamel.yaml import YAML

# Initialize YAML parser
yaml = YAML()
yaml.preserve_quotes = True  # Preserves quotes in the YAML file
yaml.indent(mapping = 2, sequence = 2, offset = 2)

#Setup file edit path 
docker_compose_path = '../docker-compose.yml'  

# Read the docker-compose.yml file
with open(docker_compose_path, 'r') as file:
    docker_compose = yaml.load(file)






if docker_compose['services']['manager_app']['env_file'] is None:
    docker_compose['services']['manager_app']['env_file']=[]
docker_compose['services']['manager_app']['env_file'] += [
     
    './S3Storage/minio.api-voditelj.shared.env'
]

#add cv-network to manager_app
if docker_compose['services']['manager_app']['networks'] is None:
    docker_compose['services']['manager_app']['networks']=[]
docker_compose['services']['manager_app']['networks'] += [
     
    'cv-network'
]



# Write the updated configuration back to docker-compose.yml
with open(docker_compose_path, 'w') as file:
    yaml.dump(docker_compose, file)

print("\ndocker-compose.yml has been updated successfully.")



docker-compose.yml has been updated successfully.


# Kreiranje config kontenjera za setup minio-a

kreiraj .yml za one-off setup kontenjer

In [3]:
%%writefile ./minio-setup-compose.yml

version: '3'

networks: 
  cv-network:
    external: true

volumes:
  minio_data:

services:
  

  minio-setup-client: # Helper to setup MinIO buckets/policies on first run
      image: minio/mc
      container_name: minio-client
      
      networks:
        - cv-network
      environment:
        MINIO_ENDPOINT: 'minio:9901'
        
      env_file:
        - ./minio.env
        - ./minio.api-voditelj.shared.env

      entrypoint: ["/scripts/minio-setup.sh"] 
      volumes:
        - ./minio-init:/config # Mount init scripts/configs here
        - ./minio-setup.sh:/scripts/minio-setup.sh

Overwriting ./minio-setup-compose.yml


Write .sh, which executes the setup code

In [1]:
%%writefile minio-setup.sh
#!/bin/sh 
echo 'Waiting for MinIO...';
until (/usr/bin/mc config host add localminio http://"${MINIO_ENDPOINT}" "${MINIO_ROOT_USER}" "${MINIO_ROOT_PASSWORD}") do sleep 1; done;
echo 'MinIO ready, setting up buckets and policies...';
/usr/bin/mc mb localminio/camera-images || true;
/usr/bin/mc anonymous set public localminio/camera-images; # TODO:  Example: make camera images public if needed, check if needed
/usr/bin/mc mb localminio/edited-images || true;
/usr/bin/mc anonymous set public localminio/edited-images; # TODO:  Example: make camera images public if needed, check if needed


echo "Creating user ${APP_MINIO_ACCESS_KEY} for API_voditelj..."
/usr/bin/mc admin user add localminio "${APP_MINIO_ACCESS_KEY}" "${APP_MINIO_SECRET_KEY}" || echo "User ${APP_MINIO_ACCESS_KEY} already exists or failed to create."
echo "Creating policy apiVoditeljPolicy..."
/usr/bin/mc admin policy create localminio apiVoditeljPolicy /config/api_voditelj_policy.json || echo "Policy apiVoditeljPolicy already exists or failed to create."
echo "Attaching policy apiVoditeljPolicy to user ${APP_MINIO_ACCESS_KEY}..."
/usr/bin/mc admin policy attach localminio apiVoditeljPolicy --user "${APP_MINIO_ACCESS_KEY}" || echo "Failed to attach policy apiVoditeljPolicy to user ${APP_MINIO_ACCESS_KEY} or already attached."
echo 'MinIO setup complete.';


Overwriting minio-setup.sh


In [29]:
!chmod +x minio-setup.sh

Write code for retention policy

In [44]:
%%writefile minio-init/lifecycle.json


{
    "Rules": [
        {
            "Expiration": {
                "Days": 30
            },
            "ID": "ExpireAfter30Days",
            "Filter": {
                "Prefix": ""
            },
            "Status": "Enabled"
        }
    ]
}

Overwriting minio-init/lifecycle.json


Write code for api_voditelj ruleset

In [5]:
%%writefile minio-init/api_voditelj_policy.json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:ListBucket",
        "s3:PutLifecycleConfiguration",
        "s3:GetLifecycleConfiguration",
         "s3:PutObject",
         "s3:GetObject",
         "s3:DeleteObject",
          "s3:HeadBucket",
          "s3:GetBucketLocation"
      ],
      "Resource": [
        "arn:aws:s3:::safe-storage-parking-lot-*"
      ]
    },
    {
      "Effect": "Allow",
      "Action": [
        "s3:CreateBucket" 
      ],
      "Resource": "arn:aws:s3:::*" 
    },
    {
      "Effect": "Allow",
      "Action": [
        "s3:ListAllMyBuckets" 
      ],
      "Resource": "arn:aws:s3:::*"
    }
  ]
}

Overwriting minio-init/api_voditelj_policy.json


Run one off container (but make sure that minio is active first)

In [6]:
!docker compose -f minio-setup-compose.yml run --rm minio-setup-client

WARN[0000] /home/benjamin/Documents/ParkMan/S3Storage/minio-setup-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
Waiting for MinIO...
]11;?\mc: Configuration written to `/root/.mc/config.json`. Please update your access credentials.
mc: Successfully created `/root/.mc/share`.
mc: Initialized share uploads `/root/.mc/share/uploads.json` file.
mc: Initialized share downloads `/root/.mc/share/downloads.json` file.
Added `localminio` successfully.
MinIO ready, setting up buckets and policies...
]11;?\mc: <ERROR> Unable to make bucket `localminio/camera-images`. Your previous request to create the named bucket succeeded and you already own it.
]11;?\Access permission for `localminio/camera-images` is set to `public`
]11;?\mc: <ERROR> Unable to make bucket `localminio/edited-images`. Your previous request to create the named bucket succeeded and you already own it.
]11;?\Access permission for `localminio/edited-i